# FunderWonder

---
## Setup

Run the cell below to install the required packages. You may see some warnings or dependency messages; these are safe to ignore.

In [ ]:
 # Install required packages (this may take a minute, and you may ignore the errors)
!pip install -qU langchain-google-genai
!pip -q install google-api-python-client google-auth google-auth-httplib2 google-auth-oauthlib
!pip install langsmith langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.6 MB/s eta 0:00:00


In [ ]:
import os

from google.colab import userdata

os.environ["LANGSMITH_TRACING"] = "true"

os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")

os.environ["LANGSMITH_PROJECT"] = "FunderWonderr"

In [ ]:
# Configure your API keys

import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
#os.environ["GCP_CREDENTIALS"] = userdata.get("GCP_CREDENTIALS")
print("API keys configured successfully!")

API keys configured successfully!


### Step 1: Create the LLM

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

---
## Part A: Implement Grants.Gov API Tool


In [ ]:
import requests
from langchain.tools import tool

# Common headers to avoid being blocked by government APIs
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

@tool
def search_grants(keywords: str) -> str:
    """
    Searches Grants.gov for open and forecasted grants.
    Use this to find a list of potential grant opportunities.
    """
    url = "https://api.grants.gov/v1/api/search2"

    payload = {
        "keyword": keywords,
        "oppStatuses": "posted|forecasted",
        "rows": 10
    }

    try:
        response = requests.post(url, json=payload, headers=HEADERS)

        if response.status_code == 200:
            results = response.json()

            # Extract opportunities from nested structure
            data = results.get("data", {})
            opportunities = data.get("oppHits") or []

            output = ""
            for opp in opportunities:
                output += f"ID: {opp.get('id')} | Number: {opp.get('number')} | Title: {opp.get('title')}\n"

            return output if output else "No grants found for these keywords."
        else:
            return f"Error: Received status code {response.status_code}"
    except Exception as e:
        return f"Error searching grants: {str(e)}"


@tool
def get_grant_details(opportunity_id: str) -> str:
    """
    Fetches the full description and synopsis for a specific grant ID.
    Use this to see if a grant covers specific costs like travel or equipment.
    """
    url = "https://api.grants.gov/v1/api/fetchOpportunity"
    payload = {"opportunityId": opportunity_id}

    try:
        response = requests.post(url, json=payload, headers=HEADERS)

        if response.status_code == 200:
            data = response.json()

            # checking for synopsis (Posted Grants)
            synopsis = data.get("synopsis")
            if synopsis:
                #explanation first, then general description
                desc = synopsis.get("synopsisExplanation") or synopsis.get("synopsisDesc")
                if desc:
                    return desc

            # 2.checking for Forecast (Forecasted Grants)
            forecast = data.get("forecast")
            if forecast:
                desc = forecast.get("forecastExplanation") or forecast.get("forecastDesc")
                if desc:
                    return f"[Forecasted Grant] {desc}"

            return "No detailed description available in synopsis or forecast sections."
        else:
            return f"Error: Could not fetch details for ID {opportunity_id}."
    except Exception as e:
        return f"Error fetching details: {str(e)}"

In [ ]:
agent_prompt = """You are FunderWonder, a helpful research assistant seeking funding opportunities.

Your Goal:
Help researchers find grants that match their needs using the Grants.gov API.

Tools available:
1. search_grants(keywords): Searches for grants. ALWAYS use this first when a user asks for funding.
2. get_grant_details(opportunity_id): Gets full details for a specific grant ID.

Process:
- When a user asks for grants on a topic, immediately call `search_grants` with relevant keywords.
- Do not ask clarifying questions before searching unless the request is completely empty.
- Once you get a list of grants, summarize them for the user.
- If the user asks for more details on a specific one, use `get_grant_details`.
"""

grant_agent = create_agent(
    model=llm,
    tools=[search_grants, get_grant_details],
    system_prompt=agent_prompt
)

In [ ]:
# test_query = "Find cancer research grants and summarize them"

# for chunk in grant_agent.stream(
#     {"messages": [{"role": "human", "content": test_query}]},
#     stream_mode="updates"
# ):
#     for step, data in chunk.items():
#         print(f"Step: {step}")
#         print(f"Content: {data['messages'][-1].content_blocks}")
#         print()
import ipywidgets as widgets
from IPython.display import display,HTML

#chat interface with html
header = widgets.HTML("""
<div style="background: linear-gradient(135deg, #15803d, #22c55e);
            padding: 20px 24px; border-radius: 12px 12px 0 0; margin-bottom: 0;">
  <h1 style="color: white; margin: 0; font-family: 'Space Grotesk', sans-serif;
             font-size: 1.6rem; letter-spacing: -0.5px;"> FunderWonder</h1>
  <p style="color: #bbf7d0; margin: 4px 0 0; font-size: 0.85rem;">
      An agent meant to aid you with finding grants!
  </p>
</div>
""")

chat_output = widgets.Output(layout=widgets.Layout(
    border='1px solid #bbf7d0',
    min_height='300px', max_height='500px',
    overflow_y='auto', padding='16px',
    background_color='#f0faf4'
))

text_input = widgets.Text(
    placeholder='e.g. "Find cancer research grants..."',
    layout=widgets.Layout(width='80%', height='40px')
)
text_input.style.description_width = '0px'

send_btn = widgets.Button(
    description='Search 🔍',
    style=widgets.ButtonStyle(button_color='#22c55e'),
    layout=widgets.Layout(width='18%', height='40px')
)

def on_send(b):
    query = text_input.value.strip() #getting query from search box
    if not query: return #if empty then exit function
    text_input.value = '' #clear input box after getting query
    with chat_output:
        display(HTML(f'<div style="background:#dcfce7;padding:10px 14px;border-radius:8px;margin:6px 0;"><b>You:</b> {query}</div>'))
        response = ""
        for chunk in grant_agent.stream({"messages": [{"role": "human", "content": query}]}, stream_mode="updates"):
            for step, data in chunk.items():
                msg = data['messages'][-1]
                if hasattr(msg, 'content') and isinstance(msg.content, str): #send the user's query to the grant agent and stream the response
                    response += msg.content
        display(HTML(f'<div style="background:white;border-left:3px solid #22c55e;padding:10px 14px;border-radius:0 8px 8px 0;margin:6px 0;"><b> FunderWonder:</b><br>{response}</div>'))

send_btn.on_click(on_send)

ui = widgets.VBox([
    header,
    chat_output,
    widgets.HBox([text_input, send_btn], layout=widgets.Layout(padding='8px', background_color='#f0faf4', border='1px solid #bbf7d0', border_top='none'))
], layout=widgets.Layout(border_radius='12px', overflow='hidden', max_width='760px'))

display(ui)

---
## Part B: Authorize Google Drive/Google Docs

In [ ]:
from google.colab import files
from google_auth_oauthlib.flow import InstalledAppFlow

SCOPES = [
    "https://www.googleapis.com/auth/documents",
    "https://www.googleapis.com/auth/drive.file"
]

uploaded = files.upload()
filename = list(uploaded.keys())[0]

flow = InstalledAppFlow.from_client_secrets_file(
    filename,
    scopes=SCOPES,
    redirect_uri="urn:ietf:wg:oauth:2.0:oob"
)

auth_url, _ = flow.authorization_url(prompt='consent')

print("Go to this URL and authorize:")
print(auth_url)

code = input("Paste the authorization code here: ")

flow.fetch_token(code=code)

creds = flow.credentials

Saving client_secret_274171314687-9gks9m4cgatjrij0pi8r57pqm5jvpgh6.apps.googleusercontent.com.json to client_secret_274171314687-9gks9m4cgatjrij0pi8r57pqm5jvpgh6.apps.googleusercontent.com (1).json
Go to this URL and authorize:
https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=274171314687-9gks9m4cgatjrij0pi8r57pqm5jvpgh6.apps.googleusercontent.com&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdocuments+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.file&state=Wrw5g5Mec13kcXiPxLMDhzUqCIw5Jr&code_challenge=Chn0h1b93F2phCi6RG4vJWEd-3b-2Rm_7TJLAKSJ1c4&code_challenge_method=S256&prompt=consent&access_type=offline
Paste the authorization code here: 4/1AfrIepA9AAqR1QnLmQT49O3V0tpBhb_1QIxow_NYG6rYTzqx_sbXhD6Fm9E


---
## Part C: Implement Google Docs API Tool

In [ ]:
from googleapiclient.discovery import build

# Build the Docs service using your active credentials
docs_service = build('docs', 'v1', credentials=creds)

@tool
def create_proposal_doc(title: str, content: str) -> str:
    """
    Creates a new Google Doc with the specified title and writes the content into it.
    Returns the URL of the created document.
    """
    try:
        #creating a blank document
        doc = docs_service.documents().create(body={'title': title}).execute()
        doc_id = doc.get('documentId')

        #inserting the text into the document
        # must use index 1, index 0 is part of doc structure
        requests = [
            {
                'insertText': {
                    'location': {'index': 1},
                    'text': content
                }
            }
        ]
        docs_service.documents().batchUpdate(documentId=doc_id, body={'requests': requests}).execute()

        return f"Document created successfully! URL: https://docs.google.com/document/d/{doc_id}/edit"
    except Exception as e:
        return f"Error creating Google Doc: {str(e)}"

In [ ]:
agent_prompt = """You are FunderWonder, a helpful research assistant seeking funding opportunities.

Your Goal:
Help researchers find grants that match their needs using the Grants.gov API.

Tools available:
1. search_grants(keywords): Searches for grants. ALWAYS use this first when a user asks for funding.
2. get_grant_details(opportunity_id): Gets full details for a specific grant ID.
3. create_proposal_doc(title, content): Creates a new Google Doc. Use this to save drafted proposals.

Process:
- When a user asks for grants, call `search_grants`.
- MANDATORY: When summarizing the list, ALWAYS include the numeric ID for every grant (e.g., "ID: 360918").
- If the user likes a grant, offer to draft a proposal.
- Once a proposal is drafted in the chat, ask the user if they want to save it to their Google Docs.
- ONLY call `create_proposal_doc` if the user explicitly asks to save it.
"""

grant_agent = create_agent(
    model=llm,
    tools=[search_grants, get_grant_details, create_proposal_doc],
    system_prompt=agent_prompt
)

In [ ]:
test_title = "FunderWonder Test Document"
test_content = "This is a test to verify the Google Docs tool is working correctly."

# Call the function directly
result = create_proposal_doc.run({"title": test_title, "content": test_content})
print(result)

Document created successfully! URL: https://docs.google.com/document/d/1uCuDbnxYrpVL2y1nG2QrPeUPtkKin-qEy7HaR1my9eU/edit


Test prompts:

1.   List item
2.   List item


